# 完整建模流程演示 (Complete Workflow)

本notebook演示hscredit库完整的风控建模流程，从数据探索到模型训练的完整Pipeline。

In [1]:
# 添加项目路径
import sys
import os
sys.path.append('../')

# 初始化设置
from hscredit.utils import init_setting
init_setting(seed=42)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import os

# 加载数据
data_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/hscredit.xlsx'
df = pd.read_excel(data_path)
print(f"数据形状: {df.shape}")
print(f"\n列名: {df.columns.tolist()}")
print(f"\n目标分布:")
print(df['FPD15'].value_counts())

数据形状: (12448, 85)

列名: ['MOB1', 'MOB2', 'loan_date', 'bj_qy24', 'mobtech80', 'bairong_v1', 'xiaoniu_c3', 'dxm_v6pro', 'bcf_fpv1', 'apply_for_admission_confidence', 'apply_for_admission_score', 'charging_fail_count_12m', 'charging_fail_count_1m', 'charging_fail_count_24m', 'charging_fail_count_3m', 'charging_fail_count_6m', 'consumer_finance_lender_count_12m', 'consumer_finance_lender_count_24m', 'consumer_finance_loan_confidence', 'consumer_finance_loan_credit_line', 'consumer_finance_loan_credit_line_avg', 'consumer_finance_loan_credit_line_max', 'consumer_finance_loan_lender_count', 'consumer_finance_loan_product_count', 'credit_loan_duration', 'hit_consumer_finance_lender_count', 'hit_lender_count', 'hit_network_loan_lender_count', 'last_performance_days', 'lender_count_12m', 'lender_count_1m', 'lender_count_24m', 'lender_count_3m', 'lender_count_6m', 'lender_inquiry_count', 'lender_inquiry_count_1m', 'lender_inquiry_count_3m', 'lender_inquiry_count_6m', 'loan_amt_between_1k_3k_coun

## Step 1: 数据准备

In [2]:
# 定义目标列和排除列
target_col = 'FPD15'
exclude_cols = ['MOB1', 'MOB2', 'loan_date', 'FPD15', 'SFPD15']

# 获取特征列
feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"特征数量: {len(feature_cols)}")

# 划分训练集和测试集
from sklearn.model_selection import train_test_split

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"训练集: {X_train.shape}, 坏样本率: {y_train.mean()*100:.2f}%")
print(f"测试集: {X_test.shape}, 坏样本率: {y_test.mean()*100:.2f}%")

特征数量: 80
训练集: (8713, 80), 坏样本率: 6.63%
测试集: (3735, 80), 坏样本率: 6.64%


## Step 2: 特征分箱

In [12]:
from hscredit.core.binning import OptimalBinning

select_features = feature_cols[:10]

# 对所有特征进行分箱
n_bins = 5
binning_results = {}

binner = OptimalBinning(method='quantile', max_n_bins=n_bins)
binner.fit(X_train[select_features], y_train)

print(f"完成 {len(select_features)} 个特征的分箱")

完成 10 个特征的分箱


## Step 3: WOE编码

In [13]:
from hscredit.core.encoders import WOEEncoder

# 使用分箱结果进行WOE编码
woe_encoder = WOEEncoder()
X_train_woe = woe_encoder.fit_transform(X_train[select_features], y_train)
X_test_woe = woe_encoder.transform(X_test[select_features])

print(f"WOE编码后形状: {X_train_woe.shape}")
print(f"\nWOE编码结果（部分）:")
print(X_train_woe.head())

WOE编码后形状: (8713, 10)

WOE编码结果（部分）:
       bj_qy24  mobtech80  bairong_v1  xiaoniu_c3  dxm_v6pro  bcf_fpv1  apply_for_admission_confidence  apply_for_admission_score  charging_fail_count_12m  charging_fail_count_1m
1138   -0.1988     0.3502     -1.0317     -0.6952    -1.0317    0.1921                          0.0155                     0.7448                  -0.0344                  0.0944
12293  -0.4282     0.1064     -0.6952      0.4499    -0.8494   -0.0021                          0.0009                     0.6500                   0.4034                  0.0414
2291    0.9562     0.3502     -1.9480     -0.4076     0.0000    0.0000                         -0.0864                     0.7146                   0.2881                  0.0414
8505   -0.2208     0.3502     -1.9480     -0.9829     0.0000    0.0000                          0.0736                    -0.1746                  -0.6487                 -0.6952
4237   -0.1844     0.1064     -1.0317      0.6169     0.0000    0.0000

## Step 4: 特征筛选

In [14]:
from hscredit.core.selectors import IVSelector, CorrSelector

# IV筛选
iv_selector = IVSelector(threshold=0.02)
X_train_iv = iv_selector.fit_transform(X_train_woe, y_train)
X_test_iv = iv_selector.transform(X_test_woe)

print(f"IV筛选前特征数: {X_train_woe.shape[1]}")
print(f"IV筛选后特征数: {X_train_iv.shape[1]}")

# 相关性筛选
corr_selector = CorrSelector(threshold=0.8)
X_train_final = corr_selector.fit_transform(X_train_iv, y_train)
X_test_final = corr_selector.transform(X_test_iv)

print(f"相关性筛选后特征数: {X_train_final.shape[1]}")

IV筛选前特征数: 10
IV筛选后特征数: 10
相关性筛选后特征数: 10


## Step 5: 模型训练

In [15]:
from hscredit.core.models import LogisticRegression
from hscredit.core.metrics import KS, AUC, Gini

# 逻辑回归模型
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_final, y_train)

# 预测
y_train_proba = lr_model.predict_proba(X_train_final)[:, 1]
y_test_proba = lr_model.predict_proba(X_test_final)[:, 1]

# 评估
train_ks = KS(y_train, y_train_proba)
train_auc = AUC(y_train, y_train_proba)
train_gini = Gini(y_train, y_train_proba)

test_ks = KS(y_test, y_test_proba)
test_auc = AUC(y_test, y_test_proba)
test_gini = Gini(y_test, y_test_proba)

print("模型性能:")
print(f"{'指标':<10} {'训练集':<10} {'测试集':<10}")
print(f"{'KS':<10} {train_ks:<10.4f} {test_ks:<10.4f}")
print(f"{'AUC':<10} {train_auc:<10.4f} {test_auc:<10.4f}")
print(f"{'Gini':<10} {train_gini:<10.4f} {test_gini:<10.4f}")

模型性能:
指标         训练集        测试集       
KS         0.9581     0.1787    
AUC        0.9915     0.5122    
Gini       0.9830     0.0243    


## Step 6: 生成评分

In [18]:
from hscredit.core.models import ScoreCard

# 创建评分卡
scorecard = ScoreCard(
    binner=binner,
    encoder=woe_encoder,
    lr_model=lr_model,
    pdo=50,
    rate=0.5,
    score=600,
    floor=300,
    ceiling=800
)

# 生成评分
train_scores = scorecard.predict(X_train_final)
test_scores = scorecard.predict(X_test_final)

print("评分统计:")
print(f"{'数据集':<10} {'最小值':<10} {'最大值':<10} {'均值':<10} {'标准差':<10}")
print(f"{'训练集':<10} {train_scores.min():<10.2f} {train_scores.max():<10.2f} {train_scores.mean():<10.2f} {train_scores.std():<10.2f}")
print(f"{'测试集':<10} {test_scores.min():<10.2f} {test_scores.max():<10.2f} {test_scores.mean():<10.2f} {test_scores.std():<10.2f}")

评分统计:
数据集        最小值        最大值        均值         标准差       
训练集        -1355.22   -1343.81   -1352.07   5.10      
测试集        -1355.22   -1343.81   -1352.13   5.07      


## Step 7: 分箱分段统计

In [19]:
# 评分分段
def score_binning(scores, n_bins=10):
    bins = np.linspace(scores.min(), scores.max(), n_bins + 1)
    return np.digitize(scores, bins[:-1])

train_bins = score_binning(train_scores)
test_bins = score_binning(test_scores)

# 统计每个分段的样本数和坏账率
score_stats = pd.DataFrame({
    '分段': range(1, 11),
    '训练样本数': [np.sum(train_bins == i) for i in range(1, 11)],
    '训练坏账率': [y_train[train_bins == i].mean() for i in range(1, 11)],
    '测试样本数': [np.sum(test_bins == i) for i in range(1, 11)],
    '测试坏账率': [y_test[test_bins == i].mean() for i in range(1, 11)]
})

print("评分分段统计:")
print(score_stats.to_string(index=False))

评分分段统计:
 分段  训练样本数  训练坏账率  测试样本数  测试坏账率
  1   6307 0.0604   2724 0.0624
  2      0    NaN      0    NaN
  3      0    NaN      0    NaN
  4      0    NaN      0    NaN
  5      0    NaN      0    NaN
  6      0    NaN      0    NaN
  7      0    NaN      0    NaN
  8      0    NaN      0    NaN
  9      0    NaN      0    NaN
 10   2406 0.0819   1011 0.0772


## Step 8: 保存完整流程结果

In [ ]:
from hscredit.utils import save_pickle
from hscredit.report.excel import ExcelWriter

# 创建输出目录
output_dir = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/workflow_results'
os.makedirs(output_dir, exist_ok=True)

# 保存模型
save_pickle(lr_model, os.path.join(output_dir, 'lr_model.pkl'))
save_pickle(woe_encoder, os.path.join(output_dir, 'woe_encoder.pkl'))
save_pickle(scorecard, os.path.join(output_dir, 'scorecard.pkl'))

# 保存结果到Excel - 使用hscredit的ExcelWriter和DataFrame.save()
output_path = os.path.join(output_dir, 'workflow_results.xlsx')

with ExcelWriter(theme_color='2639E9').set_filename(output_path) as writer:
    # 模型性能对比
    pd.DataFrame({
        '指标': ['KS', 'AUC', 'Gini'],
        '训练集': [train_ks, train_auc, train_gini],
        '测试集': [test_ks, test_auc, test_gini]
    }).save(writer, sheet_name='模型性能', title='模型性能对比', auto_width=True, condition_cols=['训练集', '测试集'])

    # 评分分段统计
    score_stats.save(writer, sheet_name='评分分段', title='评分分段统计', auto_width=True, condition_cols=['训练样本数', '测试样本数'])

    # 训练评分
    pd.DataFrame({
        '样本ID': range(len(train_scores)),
        '训练评分': train_scores
    }).save(writer, sheet_name='训练评分', title='训练集评分分布', auto_width=True)

    # 测试评分
    pd.DataFrame({
        '样本ID': range(len(test_scores)),
        '测试评分': test_scores
    }).save(writer, sheet_name='测试评分', title='测试集评分分布', auto_width=True)

print(f"完整流程结果已保存至: {output_dir}")